# Natural Entity Recognition: Model Training

This notebook fine-tunes the `bert-base-cased` model for NER using the gold-standard annotated corpus (`data/annotations/final_annotation.csv`).

Due to significant class imbalance (e.g., a high frequency of `WEATHER` entities versus scarce `NATURE` entities), 5-fold stratified cross-validation is used to ensure a robust evaluation of model performance.

**Prerequisite:** `setup.ipynb` must be executed prior to running this notebook to cache the `bert-base-cased` checkpoint locally.

In [1]:
import os
import csv
import sys
import random
from collections import Counter

import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore", message=".*pin_memory.*")

In [15]:
NOTEBOOK_DIR = os.path.abspath(os.getcwd())

if os.path.basename(NOTEBOOK_DIR) == "notebooks":
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
else:
    PROJECT_ROOT = NOTEBOOK_DIR

RANDOM_SEED = 42

FINAL_ANNOTATION_PATH = os.path.join(PROJECT_ROOT, "data", "annotations", "final_annotation.csv")

ENTITY_TYPES = ["FLORA", "FAUNA", "WEATHER", "LANDSCAPE", "NATURE"]
N_FOLDS = 5
MAX_WEIGHT = 20.0  # cap extreme weights to avoid gradient instability
MIN_WEIGHT = 0.3   # prevent O from being so cheap that false positives become nearly free
CONFIDENCE_THRESHOLD = 0.6  # predictions below this softmax confidence are forced to "O"

BERT_MODEL_NAME = "bert-base-cased"
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# Add project root to sys.path to securely import custom scripts
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from scripts.error_analysis import (
    extract_and_classify_fold, 
    summarize_errors, 
    export_error_samples, 
    print_qualitative_analysis
)

EVAL_DIR = os.path.join(PROJECT_ROOT, "data", "model_evaluation")
os.makedirs(EVAL_DIR, exist_ok=True)
ERROR_TAXONOMY_PATH = os.path.join(EVAL_DIR, "error_taxonomy.csv")

## Reproducibility

While `MultilabelStratifiedKFold` ensures a reproducible dataset split, stochastic training elements (e.g., classifier head initialization, dropout, batch shuffling) can still introduce variance between runs. 

To enforce complete reproducibility, `set_all_seeds()` fixes all sources of randomness. Applying a dynamic seed (`RANDOM_SEED + fold`) prior to initializing each fold's model guarantees that individual folds remain reproducible across executions while allowing each fold to begin with a distinct initialization.

## Data Loading

Gold-standard annotations are loaded and grouped by sentence.

In [3]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if torch.backends.mps.is_available():
        torch.backends.mps.deterministic = True
        torch.backends.mps.benchmark = False


set_all_seeds(RANDOM_SEED)  # baseline seed for anything before the fold loop

def load_sentences(path):
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    sentences = {}
    order = []  # preserve first-seen order of sentence_ids

    for row in rows:
        s_id = row["sentence_id"]
        if s_id not in sentences:
            sentences[s_id] = {"sentence_id": s_id, "text": row["sentence_text"], "entities": []}
            order.append(s_id)

        span = row["target_span"].strip()
        if not span:
            continue  # this row represents "no entity"

        start, end = row["start_char"].strip(), row["end_char"].strip()
        if not start or not end:
            print(f"  WARNING: {s_id} has target_span {span!r} but missing "
                  f"start_char/end_char -- skipped.")
            continue

        sentences[s_id]["entities"].append({
            "start": int(start), "end": int(end),
            "label": row["entity_type"].strip(), "span": span,
        })

    for s_id in sentences:
        sentences[s_id]["entities"].sort(key=lambda e: e["start"])

    return [sentences[s_id] for s_id in order]


sentences = load_sentences(FINAL_ANNOTATION_PATH)

n_with_entities = sum(1 for s in sentences if s["entities"])
n_total_entities = sum(len(s["entities"]) for s in sentences)

print(f"Loaded {len(sentences)} sentences")
print(f"  {n_with_entities} contain at least one entity")
print(f"  {len(sentences) - n_with_entities} contain no entities (valid negative examples)")
print(f"  {n_total_entities} entity mentions total")

Loaded 305 sentences
  238 contain at least one entity
  67 contain no entities (valid negative examples)
  795 entity mentions total


## Tokenization and Subword Alignment

The tokenizer and the 11-label BIO (Beginning, Inside, Outside) tagging scheme are initialized here. 

The `tokenize_and_align_labels()` function maps character offsets to subword tokens under the following constraints:
*   Only the **first** subword of a complete word receives a valid `B-` or `I-` label.
*   Continuation subwords and special tokens are assigned a label of `-100`.

This approach aligns evaluation metrics with whole words and ensures the PyTorch loss function ignores subword fragments during optimization.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

label_list = ["O"] + [f"{p}-{t}" for t in ENTITY_TYPES for p in ("B", "I")]
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}

print(f"{len(label_list)} labels:", label_list)


def tokenize_and_align_labels(sentence, tokenizer, label_to_id):
    encoding = tokenizer(sentence["text"], return_offsets_mapping=True, truncation=True)
    offsets = encoding["offset_mapping"]
    word_ids = encoding.word_ids()

    labels = []
    previous_word_id = None
    started_entities = set()  # indices into sentence["entities"] already given a B- tag

    for token_idx, (start, end) in enumerate(offsets):
        word_id = word_ids[token_idx]

        if word_id is None or word_id == previous_word_id:
            labels.append(-100)  # special token or a continuation subword
        else:
            matched_idx = None
            for i, ent in enumerate(sentence["entities"]):
                if start >= ent["start"] and end <= ent["end"]:
                    matched_idx = i
                    break

            if matched_idx is None:
                labels.append(label_to_id["O"])
            elif matched_idx not in started_entities:
                labels.append(label_to_id[f"B-{sentence['entities'][matched_idx]['label']}"])
                started_entities.add(matched_idx)
            else:
                labels.append(label_to_id[f"I-{sentence['entities'][matched_idx]['label']}"])

        previous_word_id = word_id

    return {"input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "labels": labels}

11 labels: ['O', 'B-FLORA', 'I-FLORA', 'B-FAUNA', 'I-FAUNA', 'B-WEATHER', 'I-WEATHER', 'B-LANDSCAPE', 'I-LANDSCAPE', 'B-NATURE', 'I-NATURE']


## Tokenization Verification

Verify that subword alignments map correctly back to original entity spans.

In [5]:
def decode_bio_to_spans(offsets, word_ids, labels, id_to_label):
    spans, current, current_word_id = [], None, None
    previous_word_id = None
    for (start, end), wid, lab_id in zip(offsets, word_ids, labels):
        if wid is None:
            previous_word_id = wid
            continue
        if wid == previous_word_id:
            # continuation subword -- ignore whatever label sits here,
            # regardless of value; only a word's FIRST subword counts
            if current is not None and wid == current_word_id:
                current["end"] = end
            previous_word_id = wid
            continue
        if lab_id != -100:
            lab = id_to_label[lab_id]
            if lab == "O":
                if current: spans.append(current); current = None
            elif lab.startswith("B-"):
                if current: spans.append(current)
                current = {"start": start, "end": end, "label": lab[2:]}
                current_word_id = wid
            elif lab.startswith("I-"):
                if current and current["label"] == lab[2:]:
                    current["end"] = end; current_word_id = wid
                else:
                    if current: spans.append(current)
                    current = {"start": start, "end": end, "label": lab[2:]}
                    current_word_id = wid
        previous_word_id = wid
    if current: spans.append(current)
    return spans


mismatches = 0
for s in sentences:
    encoding = tokenizer(s["text"], return_offsets_mapping=True, truncation=True)
    result = tokenize_and_align_labels(s, tokenizer, label_to_id)
    decoded = decode_bio_to_spans(encoding["offset_mapping"], encoding.word_ids(), result["labels"], id_to_label)

    original = sorted((e["start"], e["end"], e["label"]) for e in s["entities"])
    decoded_sorted = sorted((d["start"], d["end"], d["label"]) for d in decoded)

    if original != decoded_sorted:
        mismatches += 1
        print(f"MISMATCH in {s['sentence_id']}: {s['text']}")
        print("  original:", original, "\n  decoded: ", decoded_sorted)

print(f"\nChecked {len(sentences)} sentences - {mismatches} mismatches found.")
assert mismatches == 0, (
    f"{mismatches} sentences failed the verification - fix final_annotation.csv"
)


Checked 305 sentences - 0 mismatches found.


## Stratified K-Fold Split

The dataset is partitioned into `N_FOLDS` folds using `MultilabelStratifiedKFold`. Given the sparsity of certain classes (e.g., `NATURE` contains approximately 33 total mentions), random splitting risks isolating rare entities into a single fold. Stratification ensures a uniform distribution of all entity types across every fold.

*The output below verifies the token-level entity distribution per fold.*

In [6]:
# Multi-label indicator: for each sentence, which entity types does it contain at least one mention of?
Y = np.zeros((len(sentences), len(ENTITY_TYPES)), dtype=int)
for i, s in enumerate(sentences):
    types_present = {e["label"] for e in s["entities"]}
    for j, t in enumerate(ENTITY_TYPES):
        if t in types_present:
            Y[i, j] = 1

X = np.arange(len(sentences)).reshape(-1, 1)

mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
for fold_idx, (_, val_idx) in enumerate(mskf.split(X, Y)):
    for i in val_idx:
        sentences[i]["fold"] = fold_idx

fold_counts = [Counter() for _ in range(N_FOLDS)]
for s in sentences:
    for e in s["entities"]:
        fold_counts[s["fold"]][e["label"]] += 1

print(f"{'':12s}" + "".join(f"fold{f:<8d}" for f in range(N_FOLDS)))
for t in ENTITY_TYPES:
    print(f"{t:12s}" + "".join(f"{fold_counts[f][t]:<12d}" for f in range(N_FOLDS)))

            fold0       fold1       fold2       fold3       fold4       
FLORA       15          16          16          14          16          
FAUNA       11          9           11          10          14          
WEATHER     50          66          61          60          74          
LANDSCAPE   51          78          60          55          75          
NATURE      6           6           6           8           7           


## Class-Weighted Loss

The dataset is heavily skewed: approximately 95% of all tokens belong to the `"O"` (background) class, while the rarest class (`FAUNA`) represents less than 0.5%. Under an unweighted loss function, the model could disproportionately ignore rare classes while maintaining high overall accuracy.

A balanced inverse-frequency weighting scheme (`total / (n_classes * count)`) is computed via `compute_class_weights()` to penalize misclassifications on rare entities more heavily. A ceiling limit (`MAX_WEIGHT`) prevents early training instability caused by extreme weights.

In [7]:
def compute_class_weights(sentences, tokenizer, label_to_id):
    counts = Counter()
    for s in sentences:
        result = tokenize_and_align_labels(s, tokenizer, label_to_id)
        for lab_id in result["labels"]:
            if lab_id != -100:
                counts[lab_id] += 1

    total = sum(counts.values())
    n_classes = len(label_to_id)
    weights = torch.ones(n_classes)
    for label_id, count in counts.items():
        weights[label_id] = total / (n_classes * count)
    return weights


class_weights = compute_class_weights(sentences, tokenizer, label_to_id)
class_weights_capped = torch.clamp(class_weights, min=MIN_WEIGHT, max=MAX_WEIGHT)

print(f"{'label':12s} {'raw':>8s} {'capped':>8s}")
for i, (raw, capped) in enumerate(zip(class_weights, class_weights_capped)):
    print(f"{id_to_label[i]:12s} {raw:>8.2f} {capped:>8.2f}")

label             raw   capped
O                0.10     0.30
B-FLORA         14.01    14.01
I-FLORA         83.01    20.00
B-FAUNA         19.62    19.62
I-FAUNA        134.89    20.00
B-WEATHER        3.47     3.47
I-WEATHER       43.16    20.00
B-LANDSCAPE      3.38     3.38
I-LANDSCAPE     25.10    20.00
B-NATURE        32.70    20.00
I-NATURE        49.05    20.00


## Custom Weighted-Loss Trainer

The default Hugging Face `Trainer` computes unweighted `CrossEntropyLoss`. The `Trainer` class is overridden via custom subclassing to integrate computed class weights into the loss function.

Utilizing `ignore_index=-100` ensures special tokens and subword continuations do not contribute to loss gradients.

In [8]:
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = CrossEntropyLoss(
            weight=self.class_weights.to(logits.device),
            ignore_index=-100,
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

## Dataset Construction and Data Collation

`build_dataset()` formats tokenized records into standard Hugging Face `datasets.Dataset` objects.

`DataCollatorForTokenClassification` dynamically pads sequences within each batch. Labels are padded with `-100` rather than `0` to prevent the model from treating padding positions as valid background (`"O"`) entities.

In [9]:
def build_dataset(sentence_list, tokenizer, label_to_id):
    records = [tokenize_and_align_labels(s, tokenizer, label_to_id) for s in sentence_list]
    return Dataset.from_list(records)


data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

## Evaluation helper

Generates sequence evaluation metrics (Precision, Recall, F1) using `seqeval` on held-out fold data.

In [10]:
def evaluate_fold(trainer, eval_dataset, id_to_label):
    predictions, labels, _ = trainer.predict(eval_dataset)
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[id_to_label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id_to_label[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    report = classification_report(true_labels, true_predictions, output_dict=True, zero_division=0)
    overall = {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }
    return overall, report, predictions, labels

## Train and evaluate: full 5-fold loop

Execute the complete cross-validation pipeline: for each fold, a fresh model is instantiated, trained on the remaining four folds, and evaluated against the held-out validation set.

An `EarlyStoppingCallback(patience=2)` halts training when validation loss ceases to improve, preventing overfitting.

In [13]:
all_fold_reports = []
all_fold_overall = []
all_fold_errors = []

for fold in range(N_FOLDS):

    print(f"\n{'='*20} FOLD {fold} {'='*20}")

    set_all_seeds(RANDOM_SEED + fold)
    train_sents = [s for s in sentences if s["fold"] != fold]
    val_sents = [s for s in sentences if s["fold"] == fold]
    ds_train_fold = build_dataset(train_sents, tokenizer, label_to_id)
    ds_val_fold = build_dataset(val_sents, tokenizer, label_to_id)

    model_fold = AutoModelForTokenClassification.from_pretrained(
        BERT_MODEL_NAME,
        num_labels=len(label_list),
        id2label=id_to_label,
        label2id=label_to_id,
    )

    training_args_fold = TrainingArguments(
        output_dir=os.path.join(CHECKPOINTS_DIR, f"fold{fold}"),
        num_train_epochs=6,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",           # must match eval_strategy for load_best_model_at_end
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,              # only keep the best checkpoint, not every epoch
        logging_steps=10,
        report_to="none",
    )

    trainer_fold = WeightedLossTrainer(
        model=model_fold,
        args=training_args_fold,
        train_dataset=ds_train_fold,
        eval_dataset=ds_val_fold,
        data_collator=data_collator,
        class_weights=class_weights_capped,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer_fold.train()
    overall, report, predictions, labels = evaluate_fold(trainer_fold, ds_val_fold, id_to_label)


    print(
            f"Fold {fold} overall -> Precision: {overall['precision']:.3f} | "
            f"Recall: {overall['recall']:.3f} | F1: {overall['f1']:.3f}"
        )
    
    all_fold_overall.append(overall)
    all_fold_reports.append(report)

    
    fold_errors = extract_and_classify_fold(
        val_sentences=val_sents, 
        predictions=predictions, 
        labels=labels, 
        tokenizer=tokenizer, 
        id_to_label=id_to_label, 
        decode_fn=decode_bio_to_spans
    )
    all_fold_errors.append(fold_errors)

# average across all 5 folds - this is the real, final reported number
print(f"\n{'='*20} AVERAGED ACROSS {N_FOLDS} FOLDS {'='*20}")

avg = {m: np.mean([f[m] for f in all_fold_overall]) for m in ["precision", "recall", "f1"]}
print(f"Overall Across Folds -> Precision: {avg['precision']:.2f} | Recall: {avg['recall']:.2f} | F1: {avg['f1']:.2f}")

for ent in ENTITY_TYPES:
    p, r, f1 = np.mean([[s[ent]["precision"], s[ent]["recall"], s[ent]["f1-score"]] 
                        for s in all_fold_reports if ent in s], axis=0)
    print(f"{ent:12s} precision={p:.2f}  recall={r:.2f}  f1={f1:.2f}")


==================== FOLD 0 ====================


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,1.752200,1.227007
2,0.994900,0.670788
3,0.655800,0.487827
4,0.437800,0.443182
5,0.378600,0.404364
6,0.249100,0.401619


Fold 0 overall -> Precision: 0.633 | Recall: 0.895 | F1: 0.741

==================== FOLD 1 ====================


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss
1,1.593400,1.553079
2,0.816100,1.193218
3,0.772000,1.042934
4,0.342700,1.021995
5,0.317600,1.088857
6,0.204100,1.078135


Fold 3 overall -> Precision: 0.535 | Recall: 0.823 | F1: 0.649

==================== FOLD 4 ====================


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,1.766300,1.266453
2,1.130700,0.726513
3,0.529000,0.590333
4,0.438700,0.533651
5,0.344100,0.489785
6,0.303700,0.502037


Fold 4 overall -> Precision: 0.596 | Recall: 0.898 | F1: 0.717

==================== AVERAGED ACROSS 5 FOLDS ====================
Overall Across Folds -> Precision: 0.59 | Recall: 0.86 | F1: 0.70
FLORA        precision=0.61  recall=0.86  f1=0.71
FAUNA        precision=0.45  recall=0.83  f1=0.58
WEATHER      precision=0.66  recall=0.90  f1=0.76
LANDSCAPE    precision=0.59  recall=0.87  f1=0.70
NATURE       precision=0.29  recall=0.57  f1=0.38


## Error Taxonomy & Summary

Every prediction-to-gold-standard mismatch across all five folds is classified into one of four taxonomic categories:
*   **Label Confusion:** Correct span boundaries, incorrect entity type.
*   **Boundary Error:** Correct entity type, incorrect span edges.
*   **Spurious:** Predicted entity where none exists (false positive).
*   **Missed:** Unpredicted true entity (false negative).

Results are aggregated and exported to `error_taxonomy.csv` for analysis.

In [16]:
df_summary = summarize_errors(all_fold_errors)
display(df_summary)

_ = export_error_samples(all_fold_errors, save_path=ERROR_TAXONOMY_PATH)
print(f"Full error dataset exported to {ERROR_TAXONOMY_PATH}")

,Error Category,Total Count,Percentage
0,label_confusion,32,6.31
1,boundary_error,93,18.34
2,spurious,358,70.61
3,missed,24,4.73


Full error dataset exported to /Users/sara/Documents/GitHub/understory/data/model_evaluation/error_taxonomy.csv


In [17]:
# Prints an actual sample of spurious (false-positive) predictions for the two rarest classes
print_qualitative_analysis(
    all_fold_errors, 
    error_type="spurious", 
    focus_classes={"FAUNA", "NATURE"}, 
    sample_size=5
)

--- QUALITATIVE ANALYSIS: SPURIOUS ---
[Fold 4 | ID: s00123] I said with men, and with the thoughts of men, 60 I held but slight communion; but instead, My joy was in the wilderness,--to breathe The difficult air of the iced mountain's top,Where the birds dare not build--nor insect's wing Flit o'er the herbless granite; or to plunge Into the torrent, and to roll along On the swift whirl of the new-breaking wave Of river-stream, or Ocean, in their flow.
   DETAILS: {'text': 'wing', 'pred_label': 'FAUNA'}

[Fold 1 | ID: s00680] Her lips were red, her looks were free, Her locks were yellow as gold: Her skin was as white as leprosy, The Night-Mare LIFE-IN-DEATH was she, Who thicks man's blood with cold.
   DETAILS: {'text': 'leprosy', 'pred_label': 'FAUNA'}

[Fold 0 | ID: s00564] From my wings are shaken the dews that waken The sweet buds every one, When rocked to rest on their mother’s breast, As she dances about the sun.
   DETAILS: {'text': 'wings', 'pred_label': 'FAUNA'}

[Fold 2 | ID:

## Confidence Thresholding

Previous error analysis revealed that 66.9% of errors were spurious (false positives) caused by trusting low-confidence predictions. To mitigate this, a post-processing **Confidence Threshold** is introduced. Any prediction with a softmax probability below `CONFIDENCE_THRESHOLD = 0.6` is forced to the `"O"` (background) class. 

**Threshold Selection (Averaged across 5 folds):**
Threshold values were swept from 0.0 to 0.9 across all saved checkpoints to find the optimal balance without requiring model retraining:
*   **Baseline (No Threshold):** 0.700 F1
*   **Threshold 0.5:** 0.774 F1
*   **Threshold 0.6:** **0.778 F1** (Optimal)
*   **Threshold 0.7:** 0.776 F1

The benefit of thresholding is robust across all folds. Because performance remains flat between 0.5 and 0.7, 0.6 is reported as an evidence-based, generalized choice.

In [18]:
def evaluate_with_threshold(trainer, eval_dataset, id_to_label, threshold=CONFIDENCE_THRESHOLD):
    raw_logits, labels, _ = trainer.predict(eval_dataset)
    probs = torch.nn.functional.softmax(torch.tensor(raw_logits), dim=-1).numpy()
    predictions = np.argmax(probs, axis=2)
    confidences = np.max(probs, axis=2)

    o_id = label_to_id["O"]
    predictions = np.where(confidences < threshold, o_id, predictions)

    true_labels = [[id_to_label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id_to_label[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


def load_fold_trainer(fold_num, checkpoints_dir, tokenizer, data_collator):
    fold_dir = os.path.join(checkpoints_dir, f"fold{fold_num}")
    checkpoint_subdirs = [d for d in os.listdir(fold_dir) if d.startswith("checkpoint-")]
    checkpoint_path = os.path.join(fold_dir, sorted(checkpoint_subdirs)[-1])
    model_reloaded = AutoModelForTokenClassification.from_pretrained(checkpoint_path)
    predict_args = TrainingArguments(
        output_dir=os.path.join(CHECKPOINTS_DIR, "tmp_predict"),
        report_to="none",
        per_device_eval_batch_size=8,
    )
    return Trainer(model=model_reloaded, args=predict_args, data_collator=data_collator)


thresholded_f1s = []
for fold_num in range(N_FOLDS):
    trainer_reloaded = load_fold_trainer(fold_num, CHECKPOINTS_DIR, tokenizer, data_collator)
    val_sents_fold = [s for s in sentences if s["fold"] == fold_num]
    ds_val_reloaded = build_dataset(val_sents_fold, tokenizer, label_to_id)

    result = evaluate_with_threshold(trainer_reloaded, ds_val_reloaded, id_to_label)
    thresholded_f1s.append(result["f1"])
    print(f"Fold {fold_num} (threshold={CONFIDENCE_THRESHOLD}):", result)

print(f"\nAverage F1 with thresholding: {np.mean(thresholded_f1s):.3f}")

Fold 0 (threshold=0.6): {'precision': np.float64(0.7602739726027398), 'recall': np.float64(0.8345864661654135), 'f1': np.float64(0.7956989247311828)}


Fold 1 (threshold=0.6): {'precision': np.float64(0.7318435754189944), 'recall': np.float64(0.7485714285714286), 'f1': np.float64(0.7401129943502824)}


Fold 2 (threshold=0.6): {'precision': np.float64(0.7409638554216867), 'recall': np.float64(0.7987012987012987), 'f1': np.float64(0.7687500000000002)}


Fold 3 (threshold=0.6): {'precision': np.float64(0.743421052631579), 'recall': np.float64(0.7687074829931972), 'f1': np.float64(0.7558528428093646)}


Fold 4 (threshold=0.6): {'precision': np.float64(0.7835051546391752), 'recall': np.float64(0.8172043010752689), 'f1': np.float64(0.8)}

Average F1 with thresholding: 0.772


## Final Model Training

The definitive model is trained across the entire annotated dataset, keeping a 10% stratified slice for early stopping.

The final checkpoint and tokenizer are serialized to `checkpoints/final/saved_model`.

In [19]:
mskf_final = MultilabelStratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
final_train_idx, final_val_idx = next(mskf_final.split(X, Y))

final_train_sents = [sentences[i] for i in final_train_idx]
final_val_sents = [sentences[i] for i in final_val_idx]
print(f"Final model: {len(final_train_sents)} train sentences, "
      f"{len(final_val_sents)} monitoring-only validation sentences")

ds_train_final = build_dataset(final_train_sents, tokenizer, label_to_id)
ds_val_final = build_dataset(final_val_sents, tokenizer, label_to_id)

set_all_seeds(RANDOM_SEED)

final_model = AutoModelForTokenClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id,
)

final_training_args = TrainingArguments(
    output_dir=os.path.join(CHECKPOINTS_DIR, "final"),
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    logging_steps=10,
    report_to="none",
)

final_trainer = WeightedLossTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=ds_train_final,
    eval_dataset=ds_val_final,
    data_collator=data_collator,
    class_weights=class_weights_capped,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

final_trainer.train()

# Save for reuse
FINAL_MODEL_DIR = os.path.join(CHECKPOINTS_DIR, "final", "saved_model")
final_trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Final model saved to {FINAL_MODEL_DIR}")

Final model: 274 train sentences, 31 monitoring-only validation sentences


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,1.557300,1.219923
2,0.959500,0.844310
3,0.551100,0.751081
4,0.387200,0.750055
5,0.244200,0.780769
6,0.110200,0.767039


Final model saved to /Users/sara/Documents/GitHub/understory/checkpoints/final/saved_model
